# CYGNSS inundation anomaly visualization with population data

Import the necessary libraries for working with Google Earth Engine (GEE) and geemap.

In [1]:
import ee
import geemap
import cmocean 
import numpy as np
import matplotlib as mpl

SET PROJECT NAME, authenticate and set up your Earth Engine environment by creating a Google Cloud Project and enabling the Earth Engine API (https://www.google.com/url?q=https%3A%2F%2Fconsole.cloud.google.com%2Fapis%2Fapi%2Fearthengine.googleapis.com), if you haven’t already done so. Detailed setup instructions are available at https://developers.google.com/earth-engine/guides/access#a-role-in-a-cloud-project

In [2]:
project_name = 'YOUR_EE_PROJECTNAME'
ee.Authenticate()
ee.Initialize(project=project_name) 

Load the CYGNSS collection

In [3]:
cygnss_collection = ee.ImageCollection("projects/sat-io/open-datasets/CYGNSS/inundation")


Set up the functions to (i) remove NaN values from CYGNSS data, (ii) convert inundation data to fractional inundation, and (iii) remove bias from the fractional inundation data (in case of cygnss anomalies).

In [4]:
def mask_inundation(image):
    image = image.updateMask(image.neq(255))  # mask no-data values (255)
    return image.divide(100)  # scale to fractional values

def mask_anomalies(image):
    image = image.updateMask(image.neq(255))  # mask no-data values (255)
    image = image.subtract(100)               # remove bias
    return image.divide(100)                  # scale to fractional values

Select the date of interest for the image, filter the collection accordingly, and select the band of interest

In [5]:
date = '2025-11-17'
cygnss_image = cygnss_collection.filterDate(date, ee.Date(date).advance(1, 'day')).first() # the syntax of the function is Collection.filterDate(start, stop).first() 

inundation_anomalies = mask_anomalies(cygnss_image.select('inundation_anomaly_1day')) # selecting the appropriate band and applying a function to mask no-data values, removing bias, and scale to fractional inundation

Select the population dataset and filter it by year. As WorldPop data are available up to 2020, the most recent available year was selected. Calculate population density per 1km2

In [6]:
worldpop_2020 = (ee.ImageCollection("WorldPop/GP/100m/pop").filter(ee.Filter.eq("year", 2020)))

source_image = ee.Image(worldpop_2020.first()).select("population")
source_projection = source_image.projection()

population = (worldpop_2020.mosaic().select("population").setDefaultProjection(source_projection))

pixel_area_km2 = ee.Image.pixelArea().divide(1e6)

population_density = (population.divide(pixel_area_km2).rename("population_density"))
population_density_log = (
    population_density
    .updateMask(population_density.gte(1))
    .log10()
    .clamp(0, 4)
    .rename("log10_density")
)


Set the visualization parameters (colors and value ranges) for the plotted bands. You can define your own colormap. By default, GEE accepts HEX or CSS color codes. Separate visualization parameters are defined for inundation anomaly data (nominal range: -1 to 1) and population data.

In [7]:
# Inundation anomalies palette and visualisation params
anomaly_palette = cmocean.cm.balance_r(np.linspace(0, 1, 256)) #palette (balance) from matplotlib (https://matplotlib.org/cmocean/)
anomaly_palette_hex = [mpl.colors.rgb2hex(color) for color in anomaly_palette]

anomaly_vis = {
    'min': -0.2,  
    'max': 0.2,  
    'palette': anomaly_palette_hex  # Paleta kolorów dla anomalii (niebieski-czerwony)
}

#Population palette and visualisation params
pop_vis_log = {
    "min": 0,
    "max": 4,
    "palette": [
        "#ffffe5",
        "#fff7bc",
        "#fee391",
        "#fec44f",
        "#fe9929",
        "#ec7014",
        "#cc4c02",
        "#993404",
        "#662506"
    ]
}

Plot the region of interest with inundation anomaly and population data.

In [8]:
roi = ee.Geometry.Rectangle([
    110, -8.5,
    111, -7
])

Map = geemap.Map()
Map.add_basemap("HYBRID")
Map.addLayer(population_density_log, pop_vis_log, "Population density (2020, log scale)") # Population density in logarithmic scale
Map.addLayer(inundation_anomalies, anomaly_vis, "Inundation Anomaly")  # Add the layer with inundation anomalies using anomaly_vis params
Map.add_colorbar(anomaly_vis, label="Inundation Anomaly", orientation="vertical", position="bottomleft") # Add colorbar to the inundation anomaly visualisation
Map.add_colorbar(pop_vis_log, label="People per km² (log scale)", orientation="vertical", position="bottomright") # Add colorbar to the population visualisation
Map.center_object(roi, 10) # Set the initial view to the ROI
Map # Display the map

Map(center=[-7.749846038917108, 110.50000000000009], controls=(WidgetControl(options=['position', 'transparent…